# validation check — delta buckets (market − lrr) → blanket AWAY, early season

Split matches into `N_BUCKETS` (base 6) equal-count buckets of `delta = market − elo(lrr)`; boundaries
fit on the **train fold only**. Rule: bet the first bucket (market strongly below elo) on AWAY. Blanket model. Population = early season (gw ≤ 8).
Rough-draft partition across three notebooks: first bucket → AWAY, middle → DRAW, last bucket → HOME.
`N_BUCKETS` is a knob — swept 3..14 at the bottom. Region/outcome fixed from the full-sample look →
meta-DoF; gates test time/league robustness.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE — model space (ignored by the blanket model; kept for dropna) -----
MARKET = "mrkt_home_away_impl_diff"
ELO    = "teams_elo_diff_lrr_impl_diff"
FEATURES = [MARKET, ELO]

In [3]:
# --- TARGET ------------------------------------------------------------------
OUTCOME = "t_away_flg"
IMPLIED = "mrkt_away_impl"

In [4]:
# --- ABT FILTERING — early season (gw <= 8) ---------------------------------
MAX_GAMEWEEK = 8
matches = abt[abt["gameweek"] <= MAX_GAMEWEEK].dropna(
    subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()
print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 1544   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER — delta buckets (bet first bucket) ----------------------------
# delta = market - elo(lrr). N_BUCKETS equal-count buckets, boundaries fit on TRAIN only. N_BUCKETS is
# a knob (swept below). Contract: segmenter(train) -> labeler ; labeler(rows) -> Series (NaN = skip).
N_BUCKETS = 6

def _delta(df):
    return df[MARKET] - df[ELO]

def region_segmenter(n_buckets):
    def segmenter(train):
        lo = _delta(train).quantile(1 / n_buckets)
        def labeler(rows):
            dd = _delta(rows)
            lab = pd.Series(np.nan, index=rows.index, dtype="float64")
            lab[dd < lo] = 1.0
            return lab
        return labeler
    return segmenter

segmenter = region_segmenter(N_BUCKETS)

# Self-description (in-sample): the N_BUCKETS delta buckets; bet buckets marked.
_d = _delta(matches)
_b = pd.qcut(_d, N_BUCKETS, labels=False)
print(f"in-sample delta buckets (N_BUCKETS={N_BUCKETS}; bet first bucket -> AWAY)")
print(" bucket      n   delta    win%  impl%   edge     ROI   bet")
for b in range(N_BUCKETS):
    m = matches[_b == b]
    wr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    bet = "AWAY" if b == 0 else "  --"
    print(f" {b:>6} {len(m):>7}  {_d[_b == b].mean():+6.3f}   {wr*100:5.1f}  {im*100:5.1f}  "
          f"{(wr-im)*100:+5.1f}  {pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {bet}")

in-sample delta buckets (N_BUCKETS=6; bet first bucket -> AWAY)
 bucket      n   delta    win%  impl%   edge     ROI   bet
      0     258  -0.159    47.7   43.7   +3.9  +0.090  AWAY
      1     258  -0.066    32.9   37.5   -4.5  -0.121    --
      2     258  -0.014    27.5   33.0   -5.5  -0.167    --
      3     256  +0.030    27.3   30.6   -3.3  -0.107    --
      4     257  +0.076    24.9   26.5   -1.6  -0.061    --
      5     257  +0.164    22.2   27.6   -5.4  -0.197    --


In [6]:
# --- MODEL — blanket bettor -------------------------------------------------
class BlanketBettor:
    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        return self
    def predict_proba(self, X):
        n = len(X)
        return np.column_stack([np.zeros(n), np.ones(n)])

def make_model():
    return BlanketBettor()

BUFFER = 0.0

## Summary

In [7]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="elodelta_away_first_blanket", space="elo_delta", target="away",
              abt_filter="major, gw<=8, dropna",
              segmentation=f"delta=market-lrr, {N_BUCKETS} qcut buckets (train-fit), bet first bucket",
              segment_filter=f"first 1 of {N_BUCKETS} (blanket)")

elodelta_away_first_blanket   pooled +0.084 (260 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           PASS  (+3/4 seasons)
    2223 +0.067    2324 +0.238    2425 +0.138    2526 -0.085

GATE 2  league — ROI by league / drop best   PASS
    germany +0.215    france +0.119    spain +0.028    england +0.010    italy -0.013
    pooled without best league (germany): +0.038

GATE 3  walk-forward — ROI by test season    FAIL
    2425 +0.113    2526 -0.085


## Extra checks — below the template

In [8]:
# EXTRA CHECK — knob robustness: number of delta buckets n = 3..14 (bet first bucket -> AWAY).
# A real edge survives a range of n; one that appears at a single split is fragile.
print(" n_buckets   pooled    profit    bets   seasons+   bet_frac")
for n in range(3, 15):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=region_segmenter(n), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {n:<9} {pooled_roi(b['implied'], b['won']):+.4f}  {(b['won'] - b['implied']).sum():+8.2f}  "
          f"{len(b):>5}   +{int((ps > 0).sum())}/{len(ps)}      {1 / n:.2f}")

 n_buckets   pooled    profit    bets   seasons+   bet_frac
 3         -0.0207     -4.32    513   +1/4      0.33


 4         +0.0096     +1.56    388   +2/4      0.25
 5         +0.0214     +2.83    309   +3/4      0.20


 6         +0.0841     +9.54    260   +3/4      0.17
 7         +0.0889     +8.74    224   +3/4      0.14


 8         +0.0769     +6.86    199   +3/4      0.12
 9         +0.0668     +5.19    171   +4/4      0.11


 10        +0.0698     +5.02    154   +3/4      0.10
 11        +0.0385     +2.56    141   +3/4      0.09


 12        -0.0050     -0.30    131   +2/4      0.08
 13        -0.0077     -0.42    115   +2/4      0.08


 14        -0.0209     -1.07    108   +1/4      0.07
